In [1]:
import polars as pl 
from dbconfig import engine
print('Environtment ready!')

Environtment ready!


In [2]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(2)

polars.config.Config

In [21]:
df = pl.read_csv("monthly_nifty_50_1995_2026.csv")

In [22]:
df.describe()

statistic,Date,Price,Open,High,Low,Vol.,Change %
str,str,str,str,str,str,str,str
"""count""","""362""","""362""","""362""","""362""","""362""","""362""","""362"""
"""null_count""","""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,null,null,null,null,null,null
"""std""",null,null,null,null,null,null,null
"""min""","""01-01-1996""","""1,006.80""","""1,006.85""","""1,000.95""","""1,000.10""","""""","""-0.03%"""
"""25%""",null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null
"""max""","""01-12-2025""","""998.65""","""994.66""","""989.15""","""999.03""","""988.64M""","""9.86%"""


In [23]:
df.head()

Date,Price,Open,High,Low,Vol.,Change %
str,str,str,str,str,str,str
"""01-01-2026""","""25,320.65""","""26,173.30""","""26,373.20""","""24,919.80""","""8.38B""","""-3.10%"""
"""01-12-2025""","""26,129.60""","""26,325.80""","""26,325.80""","""25,693.25""","""5.37B""","""-0.28%"""
"""01-11-2025""","""26,202.95""","""25,696.85""","""26,310.45""","""25,318.45""","""5.50B""","""1.87%"""
"""01-10-2025""","""25,722.10""","""24,620.55""","""26,104.20""","""24,605.95""","""6.24B""","""4.51%"""
"""01-09-2025""","""24,611.10""","""24,432.70""","""25,448.95""","""24,432.70""","""5.98B""","""0.75%"""


In [24]:
df = df.rename({
    "Date": "date",
    "Price": "price",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Vol.": "volume",
    "Change %": "change_pct",
})

In [25]:
df.columns

['date', 'price', 'open', 'high', 'low', 'volume', 'change_pct']

In [26]:
df = df.with_columns(
    pl.col("date").str.to_date("%d-%m-%Y").alias("date")
)

In [27]:
df.schema

Schema([('date', Date),
        ('price', String),
        ('open', String),
        ('high', String),
        ('low', String),
        ('volume', String),
        ('change_pct', String)])

In [29]:
df.select(
        pl.min("date").alias("min_date"),
        pl.max("date").alias("max_date"),)

min_date,max_date
date,date
1995-12-01,2026-01-01


In [30]:
price_cols = ["price", "open", "high", "low"]

df = df.with_columns(
    [
        pl.col(col)
        .str.replace_all(",", "")
        .cast(pl.Float64)
        for col in price_cols
    ]
)

In [31]:
df.schema

Schema([('date', Date),
        ('price', Float64),
        ('open', Float64),
        ('high', Float64),
        ('low', Float64),
        ('volume', String),
        ('change_pct', String)])

In [32]:
df = df.with_columns(
    pl.col("change_pct")
    .str.replace("%", "")
    .cast(pl.Float64)
)

In [33]:
df.describe()

statistic,date,price,open,high,low,volume,change_pct
str,str,f64,f64,f64,f64,str,f64
"""count""","""362""",362.00,362.00,362.00,362.00,"""362""",362.00
"""null_count""","""0""",0.00,0.00,0.00,0.00,"""0""",0.00
"""mean""","""2010-12-16 05:22:12.596685""",7339.81,7284.96,7561.23,7018.17,null,1.14
"""std""",null,6793.63,6749.03,6932.55,6578.22,null,6.42
"""min""","""1995-12-01""",817.75,816.55,882.10,775.43,"""""",-26.41
"""25%""","""2003-06-01""",1480.45,1473.45,1566.50,1345.10,null,-2.58
"""50%""","""2011-01-01""",5295.55,5278.60,5499.40,5032.40,null,1.24
"""75%""","""2018-07-01""",10530.70,10479.95,10922.45,10111.30,null,5.11
"""max""","""2026-01-01""",26202.95,26325.80,26373.20,25693.25,"""988.64M""",28.07


In [35]:
df = df.with_columns(
    pl.when(pl.col("volume").str.ends_with("B"))
    .then(
        pl.col("volume")
        .str.extract(r"([\d.]+)", 1)
        .cast(pl.Float64)
        * 1_000_000_000
    )
    .when(pl.col("volume").str.ends_with("M"))
    .then(
        pl.col("volume")
        .str.extract(r"([\d.]+)", 1)
        .cast(pl.Float64)
        * 1_000_000
    )
    .otherwise(None)
    .alias("volume")
)

In [36]:
df.select('volume').head(10)

volume
f64
8380000000.00
5370000000.00
5500000000.00
6240000000.00
5980000000.00
5870000000.00
6280000000.00
7060000000.00
8550000000.00


In [37]:
df.describe()

statistic,date,price,open,high,low,volume,change_pct
str,str,f64,f64,f64,f64,f64,f64
"""count""","""362""",362.00,362.00,362.00,362.00,349.00,362.00
"""null_count""","""0""",0.00,0.00,0.00,0.00,13.00,0.00
"""mean""","""2010-12-16 05:22:12.596685""",7339.81,7284.96,7561.23,7018.17,3954576905.44,1.14
"""std""",null,6793.63,6749.03,6932.55,6578.22,3199034855.02,6.42
"""min""","""1995-12-01""",817.75,816.55,882.10,775.43,440000000.00,-26.41
"""25%""","""2003-06-01""",1480.45,1473.45,1566.50,1345.10,1520000000.00,-2.58
"""50%""","""2011-01-01""",5295.55,5278.60,5499.40,5032.40,3220000000.00,1.24
"""75%""","""2018-07-01""",10530.70,10479.95,10922.45,10111.30,5440000000.00,5.11
"""max""","""2026-01-01""",26202.95,26325.80,26373.20,25693.25,21300000000.00,28.07


In [38]:
df = df.sort("date")

In [39]:
df.select(pl.col("date").is_duplicated().sum())

date
u32
0


In [40]:
df.null_count()

date,price,open,high,low,volume,change_pct
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,13,0


In [41]:
df.head()

date,price,open,high,low,volume,change_pct
date,f64,f64,f64,f64,f64,f64
1995-12-01,908.53,867.99,927.60,854.47,null,5.39
1996-01-01,848.42,913.11,913.11,813.12,null,-6.62
1996-02-01,992.51,848.28,1067.49,848.28,null,16.98
1996-03-01,985.30,994.66,1010.50,945.07,null,-0.73
1996-04-01,1114.36,988.33,1160.16,988.33,null,13.10


In [42]:
df.tail()

date,price,open,high,low,volume,change_pct
date,f64,f64,f64,f64,f64,f64
2025-09-01,24611.10,24432.70,25448.95,24432.70,5980000000.00,0.75
2025-10-01,25722.10,24620.55,26104.20,24605.95,6240000000.00,4.51
2025-11-01,26202.95,25696.85,26310.45,25318.45,5500000000.00,1.87
2025-12-01,26129.60,26325.80,26325.80,25693.25,5370000000.00,-0.28
2026-01-01,25320.65,26173.30,26373.20,24919.80,8380000000.00,-3.10


In [43]:
nifty_monthly = df

In [44]:
nifty_monthly.write_database(
    table_name="analysis.nifty_monthly",
    connection=engine,
    if_table_exists="replace",
)

-1

In [46]:
pl.read_database(
        query = """
        select count(*)
        from analysis.nifty_monthly
        limit 5;
        """, connection = engine
        )

count
i64
362
